### Error Handling

In [1]:
try:
    res = 10 / 0
except ZeroDivisionError:
    print("You can't divide by zero")
finally:
    print("Hi, Viktor!")

You can't divide by zero
Hi, Viktor!


### Unit-Tests

In [11]:
def add(a: int, b: int) -> int:
    return a + b

def is_even(a: int) -> bool:
    return a % 2 == 0

In [18]:
import unittest

class TestMathUtils(unittest.TestCase):
    def test_add_positive_numbers(self):
        self.assertEqual(add(2, 3), 5)

    def test_add_negative_numbers(self):
        self.assertEqual(add(-1, -4), -5)

    def test_is_even_true(self):
        self.assertTrue(is_even(4))

    def test_is_even_false(self):
        self.assertFalse(is_even(3))


unittest.main(argv=['first-arg-is-ignored'], exit=0)


....
----------------------------------------------------------------------
Ran 4 tests in 0.003s

OK


### Integration tests


In [45]:
%pip install fastapi "uvicorn[standard]" requests httpx


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [46]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def root():
    return {"msg": "Hello"}


In [47]:
from fastapi.testclient import TestClient

client = TestClient(app)

def test_root():
    r = client.get("/")
    assert r.status_code == 200
    assert r.json() == {"msg": "Hello"}


### E2E tests

In [48]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()
_db: dict[int, "Todo"] = {}
_counter = 0

class Todo(BaseModel):
    id: int | None = None
    title: str

@app.post("/todos", status_code=201)
def create_todo(todo: Todo):
    global _counter
    _counter += 1
    todo.id = _counter
    _db[todo.id] = todo
    return todo

@app.get("/todos/{todo_id}")
def get_todo(todo_id: int):
    return _db[todo_id]


In [50]:
from fastapi.testclient import TestClient

client = TestClient(app)

def test_create_and_get_todo_e2e():
    create = client.post("/todos", json={"title": "Buy milk"})
    assert create.status_code == 201
    todo_id = create.json()["id"]

    get_ = client.get(f"/todos/{todo_id}")
    assert get_.status_code == 200
    assert get_.json()["title"] == "Buy milk"